# Audio Context Layer (ACL) - End-to-End Pipeline on GPU

**Instructions:**
1. In Google Colab, go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.
2. On Kaggle, ensure **Internet** is toggled **ON** in the sidebar.
3. Run the cells sequentially below.

In [ ]:
# 1. Environment and GPU Verification
!nvidia-smi
!pip install -q torch torchvision torchaudio transformers librosa soundfile reportlab pytest pyyaml scipy pandas matplotlib

In [ ]:
# 2. Clone ESC-50 Source Dataset
!git clone --depth 1 https://github.com/karolpiczak/ESC-50.git ESC-50
!rm -rf ESC-50/.git

In [ ]:
# 3. Synthesize Full Audio-Scene QA Dataset (1,600 train / 300 val / 500 test scenes)
!python -m acl.data_gen --esc50 ESC-50 --out dataset --n_train 1600 --n_val 300 --n_test 500 --seed 42

In [ ]:
# 4. Run Unit, Consistency, and Leakage Tests
!python -m pytest -q tests

In [ ]:
# 5. Train Main Model: AST (Frozen AudioSet Embeddings) + BiGRU Head
!python -m acl.train_sed --data dataset --backend ast --out runs/ast --epochs 30 --bs 32 --seed 42

In [ ]:
# 6. Train Baseline Model: Lightweight Edge CRNN
!python -m acl.train_sed --data dataset --backend crnn --out runs/crnn --epochs 40 --bs 32 --seed 42

In [ ]:
# 7. Comprehensive Quantitative Evaluation (Validation Tuning, SED F1, QA Accuracy, Bootstrapping)
!python -m acl.evaluate --data dataset --run runs/ast
!python -m acl.evaluate --data dataset --run runs/crnn

In [ ]:
# 8. Edge Feasibility & Quantization Profiling
!python -m acl.edge_report --data dataset --run runs/ast
!python -m acl.edge_report --data dataset --run runs/crnn

In [ ]:
# 9. Compile Publication-Grade Technical PDF Report
!python -m acl.make_report --data dataset --runs runs/ast runs/crnn --author "Researcher" --out report/ACL_technical_report.pdf

In [ ]:
# 10. Interactive CLI Demo on a Test Clip
!python -m acl.demo --run runs/ast --wav dataset/audio/test/test_00000.flac

In [ ]:
# 11. Bundle Artifacts for Download
!zip -qr acl_results_bundle.zip report runs/ast/results runs/crnn/results runs/ast/loss_curves.png runs/crnn/loss_curves.png runs/ast/edge.json runs/crnn/edge.json
from google.colab import files
files.download('acl_results_bundle.zip')
files.download('report/ACL_technical_report.pdf')